# Experiment No. 6
## Analyzing and Interpreting Statistical Distributions

**Aim:** Analyze and interpret probability distributions using graphical and statistical techniques.

**Dataset:** `Sub_Division_IMD_2017(1).csv`, containing Indian subdivision rainfall records by year, month, and season.

**Important note:** The supplied dataset is rainfall data, not a direct robotics sensor dataset. This notebook uses rainfall as environmental measurement data and demonstrates the statistical workflow required by the lab. For a robotics-specific submission, replace it with actual sensor readings.

**Libraries:** pandas, NumPy, Matplotlib, SciPy.

In [ ]:
%pip install pandas numpy matplotlib scipy

## 1. Load and inspect the dataset
Keep the CSV file in the same folder as this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

file_path = "Sub_Division_IMD_2017(1).csv"
df = pd.read_csv(file_path)
print('Dataset shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head())

## 2. Select a subdivision and variables

In [ ]:
subdivision = 'Madhya Maharashtra'
subset = df[df['SUBDIVISION'] == subdivision].copy()
if subset.empty:
    subdivision = df['SUBDIVISION'].dropna().iloc[0]
    subset = df[df['SUBDIVISION'] == subdivision].copy()
    print('Requested subdivision not found; using:', subdivision)
else:
    print('Analyzing subdivision:', subdivision)

variables = ['ANNUAL', 'JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']
available_variables = [c for c in variables if c in subset.columns]
data = subset[available_variables].apply(pd.to_numeric, errors='coerce')
print('Number of yearly observations:', len(subset))

## 3. Descriptive statistics
Mean, median, standard deviation, skewness, kurtosis, range, and missing values.

In [ ]:
summary = pd.DataFrame({
    'Mean': data.mean(), 'Median': data.median(),
    'Standard Deviation': data.std(), 'Variance': data.var(),
    'Skewness': data.skew(), 'Kurtosis (excess)': data.kurtosis(),
    'Minimum': data.min(), 'Maximum': data.max(),
    'Missing Values': data.isna().sum()
})
display(summary.round(3))

## 4. Histogram and fitted normal distribution

In [ ]:
annual = data['ANNUAL'].dropna()
plt.figure(figsize=(8, 5))
plt.hist(annual, bins=15, density=True, alpha=0.7, edgecolor='black', label='Observed annual rainfall')
x = np.linspace(annual.min(), annual.max(), 300)
mu, sigma = stats.norm.fit(annual)
plt.plot(x, stats.norm.pdf(x, mu, sigma), linewidth=2, label='Fitted normal distribution')
plt.title(f'Annual Rainfall Distribution — {subdivision}')
plt.xlabel('Annual rainfall (mm)'); plt.ylabel('Density'); plt.legend(); plt.tight_layout(); plt.show()

## 5. Density plots

In [ ]:
plt.figure(figsize=(9, 5))
for col in ['ANNUAL', 'JJAS', 'MAM', 'OND']:
    if col in subset.columns:
        values = pd.to_numeric(subset[col], errors='coerce').dropna()
        if len(values) > 1 and values.nunique() > 1:
            values.plot(kind='kde', label=col)
plt.title(f'Rainfall Density Curves — {subdivision}')
plt.xlabel('Rainfall (mm)'); plt.ylabel('Density'); plt.legend(); plt.tight_layout(); plt.show()

## 6. Identify potential outliers using the IQR rule

In [ ]:
q1, q3 = annual.quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = annual[(annual < lower_bound) | (annual > upper_bound)]
plt.figure(figsize=(8, 3.5)); plt.boxplot(annual, vert=False)
plt.title(f'Annual Rainfall Boxplot — {subdivision}')
plt.xlabel('Annual rainfall (mm)'); plt.tight_layout(); plt.show()
print('Potential outliers using 1.5×IQR:', len(outliers))
display(pd.DataFrame({'Year': subset.loc[outliers.index, 'YEAR'], 'Annual rainfall (mm)': outliers}).reset_index(drop=True))

## 7. Compare with a theoretical normal distribution

In [ ]:
if len(annual) >= 8 and annual.std(ddof=1) > 0:
    ks_stat, ks_pvalue = stats.kstest(annual, 'norm', args=(mu, sigma))
    print(f'Fitted normal distribution: mean={mu:.2f} mm, std={sigma:.2f} mm')
    print(f'KS statistic={ks_stat:.4f}, p-value={ks_pvalue:.4f}')
else:
    print('Not enough non-constant annual rainfall observations for this diagnostic.')
print('Note: because normal parameters are estimated from this sample, interpret the KS result as a rough diagnostic.')

## 8. Interpretation and conclusion

In [ ]:
skew = annual.skew()
if skew > 0.5:
    skew_text = 'positively skewed (longer right tail)'
elif skew < -0.5:
    skew_text = 'negatively skewed (longer left tail)'
else:
    skew_text = 'approximately symmetric by the skewness measure'
print(f'Annual rainfall for {subdivision} is {skew_text}.')
print(f'Mean = {annual.mean():.2f} mm; median = {annual.median():.2f} mm; standard deviation = {annual.std():.2f} mm.')
print('A large standard deviation indicates substantial year-to-year variation.')
print('IQR outliers are unusual observations, not automatically data errors.')
print('Conclusion: descriptive statistics, histograms, density curves, and outlier analysis help characterize measurement distributions. The same methods can be applied to real robotic sensor data.')

## Post-lab questions — short answers
1. **Probability distribution:** Describes the probabilities or relative frequencies of possible values of a random variable.
2. **Normal distribution:** A symmetric, bell-shaped distribution described by its mean and standard deviation.
3. **Skewness:** A measure of asymmetry in a distribution.
4. **Kurtosis:** A measure related to tail heaviness; pandas reports excess kurtosis.
5. **Histograms:** Show the frequency or density shape, spread, and possible unusual values.
6. **Population vs sample:** A population is the complete group of interest; a sample is a subset observed from it.
7. **Anomaly detection:** Readings far from typical values or outside expected distribution limits may be flagged for investigation.